# Map The Numbers Movie Budgets to Actor & Director Credibility

This notebook links movies in **The Numbers** budgets file  
`the_numbers_movie_budgets.csv`[file:96]  
to actor and director credibility scores computed from **The Movies Dataset** (TMDb).[web:87]

Steps:

1. Load The Numbers budget data and derive `title_clean` + `release_year`.
2. Load TMDb movie metadata and match on `title_clean` + `release_year` to get `movie_id`.
3. Use `credits.csv` to get actors and directors for those `movie_id`s.
4. Merge actor/director **credibility scores** from:
   - `actor_credibility_from_public.csv`
   - `director_credibility_from_public.csv`
5. Optionally aggregate to per‑movie features (mean / max credibility).


In [14]:
import pandas as pd
import numpy as np
import ast
import os

# Base path where all CSVs live
# BASE_PATH = "data/archive"

NUMBERS_FILE   = os.path.join("the_numbers_movie_budgets.csv")
MOVIES_FILE    = os.path.join("movies_metadata.csv")
CREDITS_FILE   = os.path.join("credits.csv")
ACTOR_CRED_FILE    = os.path.join("actor_credibility_from_public.csv")
DIRECTOR_CRED_FILE = os.path.join("director_credibility_from_public.csv")


In [15]:
import os
os.getcwd()


'/Users/saptakchakraborty/Downloads/Uni Docs/OVGU/1ST SEM/Data Science with Python/Project_IMDB/movie-success-analysis/data/actorDirectorCredibility'

## 1. Load The Numbers movie budgets

We start from `the_numbers_movie_budgets.csv`, which contains our target movie list with budget and grosses.[file:96]

To join with TMDb, we create:

- `title_clean` – lower‑cased, stripped title
- `release_year` – year extracted from `release_date`


In [16]:
numbers = pd.read_csv(NUMBERS_FILE)

numbers.head()


,rank,release_date,title,budget,domestic_gross,worldwide_gross
0,1,"Dec 16, 2015",Star Wars Ep. VII: The Force Awakens,533200000,936662225,2056046835
1,2,"Apr 23, 2019",Avengers: Endgame,400000000,858373000,2717503922
2,3,"Dec 9, 2022",Avatar: The Way of Water,400000000,688809501,2322902023
3,4,"Dec 17, 2025",Avatar: Fire and Ash,400000000,250295254,859795254
4,5,"May 17, 2025",Mission: Impossible—The Final Reckoning,400000000,197413515,591353074


In [17]:
# Normalize title and extract year for matching
numbers["title_clean"] = numbers["title"].str.strip().str.lower()
numbers["release_year"] = pd.to_datetime(numbers["release_date"], errors="coerce").dt.year

numbers[["title", "release_date", "title_clean", "release_year"]].head()


,title,release_date,title_clean,release_year
0,Star Wars Ep. VII: The Force Awakens,"Dec 16, 2015",star wars ep. vii: the force awakens,2015.0
1,Avengers: Endgame,"Apr 23, 2019",avengers: endgame,2019.0
2,Avatar: The Way of Water,"Dec 9, 2022",avatar: the way of water,2022.0
3,Avatar: Fire and Ash,"Dec 17, 2025",avatar: fire and ash,2025.0
4,Mission: Impossible—The Final Reckoning,"May 17, 2025",mission: impossible—the final reckoning,2025.0


## 2. Match The Numbers titles to TMDb movie IDs

We use `movies_metadata.csv` from The Movies Dataset.[web:87]

From it we use:

- `id` → `movie_id` (TMDb movie ID)
- `title`
- `release_date` → `release_year`

We clean titles similarly and match on `title_clean + release_year` to get `movie_id` for each Numbers movie where possible.


In [18]:
meta = pd.read_csv(MOVIES_FILE, low_memory=False)

meta["movie_id"] = pd.to_numeric(meta["id"], errors="coerce")
meta["title_clean"] = meta["title"].str.strip().str.lower()
meta["release_year"] = pd.to_datetime(meta["release_date"], errors="coerce").dt.year

meta_small = meta[["movie_id", "title_clean", "release_year"]].dropna(subset=["movie_id", "release_year"])

meta_small.head()


,movie_id,title_clean,release_year
0,862.0,toy story,1995.0
1,8844.0,jumanji,1995.0
2,15602.0,grumpier old men,1995.0
3,31357.0,waiting to exhale,1995.0
4,11862.0,father of the bride part ii,1995.0


In [19]:
# Match Numbers movies to TMDb movie_id via title + year
numbers_tmdb = numbers.merge(
    meta_small,
    on=["title_clean", "release_year"],
    how="left"
)

print("Total Numbers movies:", len(numbers_tmdb))
print("Matched to TMDb movie_id:", numbers_tmdb["movie_id"].notna().sum())
print("Match rate:", numbers_tmdb["movie_id"].notna().mean())
numbers_tmdb.head()


Total Numbers movies: 6753
Matched to TMDb movie_id: 4166
Match rate: 0.6169110025173997


,rank,release_date,title,budget,domestic_gross,worldwide_gross,title_clean,release_year,movie_id
0,1,"Dec 16, 2015",Star Wars Ep. VII: The Force Awakens,533200000,936662225,2056046835,star wars ep. vii: the force awakens,2015.0,NaN
1,2,"Apr 23, 2019",Avengers: Endgame,400000000,858373000,2717503922,avengers: endgame,2019.0,NaN
2,3,"Dec 9, 2022",Avatar: The Way of Water,400000000,688809501,2322902023,avatar: the way of water,2022.0,NaN
3,4,"Dec 17, 2025",Avatar: Fire and Ash,400000000,250295254,859795254,avatar: fire and ash,2025.0,NaN
4,5,"May 17, 2025",Mission: Impossible—The Final Reckoning,400000000,197413515,591353074,mission: impossible—the final reckoning,2025.0,NaN


## 3. Get actors and directors for matched movies

Using `credits.csv` from The Movies Dataset,[web:87] we:

- parse the JSON `cast` and `crew` lists,
- keep only credits for movies that appear in `numbers_tmdb` (have `movie_id`),
- build:
  - `actors_for_numbers`: rows of `(movie_id, person_id, person_name)` for actors
  - `directors_for_numbers`: rows of `(movie_id, person_id, person_name)` for directors


In [20]:
credits = pd.read_csv(CREDITS_FILE, low_memory=False)

def parse_json_list(s):
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

credits["cast_parsed"] = credits["cast"].apply(parse_json_list)
credits["crew_parsed"] = credits["crew"].apply(parse_json_list)

# Restrict credits to movies that appear in Numbers (with a match)
movie_ids = numbers_tmdb["movie_id"].dropna().astype(int).unique()
credits_subset = credits[credits["id"].isin(movie_ids)].copy()

len(credits_subset)


4165

In [21]:
actor_rows = []
director_rows = []

for _, row in credits_subset.iterrows():
    mid = row["id"]
    # Cast = actors
    for p in row["cast_parsed"]:
        pid = p.get("id")
        name = p.get("name")
        if pid is not None:
            actor_rows.append((mid, pid, name))
    # Crew = find directors
    for p in row["crew_parsed"]:
        if p.get("job") == "Director":
            pid = p.get("id")
            name = p.get("name")
            if pid is not None:
                director_rows.append((mid, pid, name))

actors_for_numbers = pd.DataFrame(actor_rows, columns=["movie_id", "person_id", "person_name"])
directors_for_numbers = pd.DataFrame(director_rows, columns=["movie_id", "person_id", "person_name"])

actors_for_numbers.head(), directors_for_numbers.head()


(   movie_id  person_id    person_name
 0       862         31      Tom Hanks
 1       862      12898      Tim Allen
 2       862       7167    Don Rickles
 3       862      12899     Jim Varney
 4       862      12900  Wallace Shawn,
    movie_id  person_id      person_name
 0       862       7879    John Lasseter
 1      8844       4945     Joe Johnston
 2     11860       2226   Sydney Pollack
 3       710      10702  Martin Campbell
 4      9087       3026       Rob Reiner)

## 4. Attach actor and director credibility scores

We now merge the actor/director person IDs with the credibility tables generated earlier:

- `actor_credibility_from_public.csv`
- `director_credibility_from_public.csv`

Each actor/director attached to a Numbers movie gets their corresponding credibility metrics (e.g., `actor_credibility_score`, `director_credibility_score`).


In [22]:
actor_cred = pd.read_csv(ACTOR_CRED_FILE)
director_cred = pd.read_csv(DIRECTOR_CRED_FILE)

# Join actor credibility
actors_for_numbers = actors_for_numbers.merge(
    actor_cred,
    on=["person_id", "person_name"],
    how="left"
)

# Join director credibility
directors_for_numbers = directors_for_numbers.merge(
    director_cred,
    on=["person_id", "person_name"],
    how="left"
)

actors_for_numbers.head(), directors_for_numbers.head()


(   movie_id  person_id    person_name  actor_avg_box_office  \
 0       862         31      Tom Hanks          1.380020e+08   
 1       862      12898      Tim Allen          1.139719e+08   
 2       862       7167    Don Rickles          7.232329e+07   
 3       862      12899     Jim Varney          5.543431e+07   
 4       862      12900  Wallace Shawn          5.486260e+07   
 
    actor_median_box_office  actor_max_box_office  actor_movie_count  \
 0               36381187.5          1.066970e+09               72.0   
 1                      0.0          1.066970e+09               29.0   
 2                      0.0          1.066970e+09               29.0   
 3                      0.0          4.973669e+08               20.0   
 4                      0.0          1.066970e+09               75.0   
 
    actor_hit_rate  actor_credibility_score  
 0        0.388889                11.396148  
 1        0.206897                10.576660  
 2        0.379310                10.86649

## 5. Aggregate credibility to movie‑level features (optional)

For modelling, we often want **one row per movie** with cast/director credibility.

We aggregate actor and director scores per `movie_id`:

- Actors:
  - `cast_mean_credibility`
  - `cast_max_credibility`
  - `cast_actors_with_score` (how many actors had a score)

- Directors:
  - `director_mean_credibility`
  - `director_max_credibility`
  - `directors_with_score`


In [23]:
# Actor-level aggregation
actor_movie_features = actors_for_numbers.groupby("movie_id").agg(
    cast_mean_credibility=("actor_credibility_score", "mean"),
    cast_max_credibility=("actor_credibility_score", "max"),
    cast_actors_with_score=("actor_credibility_score", "count"),
).reset_index()

# Director-level aggregation
director_movie_features = directors_for_numbers.groupby("movie_id").agg(
    director_mean_credibility=("director_credibility_score", "mean"),
    director_max_credibility=("director_credibility_score", "max"),
    directors_with_score=("director_credibility_score", "count"),
).reset_index()

actor_movie_features.head(), director_movie_features.head()


(   movie_id  cast_mean_credibility  cast_max_credibility  \
 0         5               9.504690             10.594521   
 1        12              10.885897             12.650596   
 2        13              11.080123             13.140419   
 3        14              10.444932             12.020988   
 4        16               9.857568             11.056055   
 
    cast_actors_with_score  
 0                      22  
 1                      20  
 2                      39  
 3                      24  
 4                      17  ,
    movie_id  director_mean_credibility  director_max_credibility  \
 0         5                   9.030962                 12.054435   
 1        12                  12.000583                 12.000583   
 2        13                  11.303958                 11.303958   
 3        14                  11.160838                 11.160838   
 4        16                   9.373934                  9.373934   
 
    directors_with_score  
 0            

## 6. Final enriched Numbers dataset

Finally, we merge the movie‑level credibility features back into the `numbers_tmdb` table using `movie_id`.

The resulting DataFrame contains:

- The Numbers fields (budget, gross, etc.)[file:96]
- TMDb `movie_id`
- Actor credibility features
- Director credibility features


In [24]:
# numbers_enriched = numbers_tmdb.merge(actor_movie_features, on="movie_id", how="left")
# numbers_enriched = numbers_enriched.merge(director_movie_features, on="movie_id", how="left")

# numbers_enriched.head()

# --- 1. build a human-readable 'actors' column per movie ---

# sort actors by name so the list is stable
actors_for_numbers_sorted = actors_for_numbers.sort_values(
    ["movie_id", "person_name"]
)

actors_list = (
    actors_for_numbers_sorted
    .groupby("movie_id")["person_name"]
    .apply(lambda names: ", ".join(names.unique()))
    .reset_index()
    .rename(columns={"person_name": "actors"})
)

# --- 2. aggregate credibility per movie (as before) ---

actor_movie_features = actors_for_numbers.groupby("movie_id").agg(
    cast_mean_credibility=("actor_credibility_score", "mean"),
    cast_max_credibility=("actor_credibility_score", "max"),
    cast_actors_with_score=("actor_credibility_score", "count"),
).reset_index()

director_movie_features = directors_for_numbers.groupby("movie_id").agg(
    director_mean_credibility=("director_credibility_score", "mean"),
    director_max_credibility=("director_credibility_score", "max"),
    directors_with_score=("director_credibility_score", "count"),
).reset_index()

# --- 3. merge everything back into Numbers table ---

numbers_enriched = numbers_tmdb.merge(actor_movie_features, on="movie_id", how="left")
numbers_enriched = numbers_enriched.merge(director_movie_features, on="movie_id", how="left")
numbers_enriched = numbers_enriched.merge(actors_list, on="movie_id", how="left")

numbers_enriched.head()


,rank,release_date,title,budget,domestic_gross,worldwide_gross,title_clean,release_year,movie_id,cast_mean_credibility,cast_max_credibility,cast_actors_with_score,director_mean_credibility,director_max_credibility,directors_with_score,actors
0,1,"Dec 16, 2015",Star Wars Ep. VII: The Force Awakens,533200000,936662225,2056046835,star wars ep. vii: the force awakens,2015.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"Apr 23, 2019",Avengers: Endgame,400000000,858373000,2717503922,avengers: endgame,2019.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,"Dec 9, 2022",Avatar: The Way of Water,400000000,688809501,2322902023,avatar: the way of water,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"Dec 17, 2025",Avatar: Fire and Ash,400000000,250295254,859795254,avatar: fire and ash,2025.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,"May 17, 2025",Mission: Impossible—The Final Reckoning,400000000,197413515,591353074,mission: impossible—the final reckoning,2025.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# Optionally save enriched dataset
OUT_FILE = os.path.join("the_numbers_with_credibility.csv")
numbers_enriched.to_csv(OUT_FILE, index=False)
print("Saved:", OUT_FILE)


Saved: the_numbers_with_credibility.csv
